# Example 10.1 — Desired Trajectory and Initial Mechanism Evaluation

**Chapter 10: Complete Example: Step-by-Step Trajectory Synthesis**

Generates the drop-shaped trajectory (n = 20), analyzes an initial four-bar mechanism with the Weierstrass solution and computes the trajectory error.

Only NumPy is used for the computation (`from numpy import *`); Matplotlib is used only for the figures.

In [1]:
import matplotlib.pyplot as plt
from numpy import *

def drop_trajectory(n=20):
    """Generates n points on the drop-shaped trajectory."""
    t = linspace(0, 2 * pi, n, endpoint=False)
    f = 1 - 0.5 * sin(t / 2)
    return 100 + 50 * cos(t) * f, 100 + 50 * sin(t) * f

def analyze_mechanism(r1, r2, r3, r4, th2):
    """Loop closure (Weierstrass half-angle), one assembly
    branch. Returns NaN where it cannot be assembled."""
    A = 2 * r4 * (r1 - r2 * cos(th2))
    B = -2 * r2 * r4 * sin(th2)
    C = r1**2 + r2**2 + r4**2 - r3**2 - 2 * r1 * r2 * cos(th2)
    disc = A**2 + B**2 - C**2
    ok = disc >= 0
    root = sqrt(where(ok, disc, 0.0))
    th4 = where(ok, 2 * arctan2(-B - root, C - A), nan)
    th3 = arctan2(r4 * sin(th4) - r2 * sin(th2),
                  r1 + r4 * cos(th4) - r2 * cos(th2))
    return th3, th4

n_pts = 20
xd, yd = drop_trajectory(n_pts)

print('Desired trajectory (first 5 points):')
print(f"{'i':>3}  {'xd':>8}  {'yd':>8}")
for i in range(5):
    print(f"{i:>3}  {xd[i]:>8.2f}  {yd[i]:>8.2f}")

# Arbitrary initial design:
# [r1, r2, r3, r4, xp, yp, theta20 (deg), x0, y0]
r1, r2, r3, r4 = 80.0, 40.0, 100.0, 90.0
xp, yp, theta20, x0, y0 = 30.0, 10.0, 0.0, 60.0, 80.0

th2 = radians(theta20) + linspace(0, 2 * pi, n_pts,
                                  endpoint=False)
th3, th4 = analyze_mechanism(r1, r2, r3, r4, th2)

# Verification: the vector loop must close (residual ~ 0)
res = abs(r2 * exp(1j * th2) + r3 * exp(1j * th3)
          - r1 - r4 * exp(1j * th4))

# Coupler point and trajectory error
xa = x0 + r2 * cos(th2) + xp * cos(th3) - yp * sin(th3)
ya = y0 + r2 * sin(th2) + xp * sin(th3) + yp * cos(th3)
errors = sqrt((xa - xd)**2 + (ya - yd)**2)

print(f'r1={r1}, r2={r2}, r3={r3}, r4={r4} mm')
print(f'Coupler: xp={xp}, yp={yp} mm; '
      f'ground pivot: ({x0}, {y0}) mm')
print(f'Maximum loop-closure residual: {res.max():.2e} mm')
print(f'Mean trajectory error: {mean(errors):.4f} mm')
print(f'Maximum error:       {max(errors):.4f} mm')

fig, ax = plt.subplots(1, 2, figsize=(6.4, 2.7))
ax[0].plot(xd, yd, 'ko-', ms=3, lw=0.8, label='Desired')
ax[0].plot(xa, ya, 's--', color='tab:red', ms=3, lw=0.8,
           label='Initial mechanism')
ax[0].set_aspect('equal')
ax[0].set_xlabel('x (mm)'); ax[0].set_ylabel('y (mm)')
ax[0].legend(fontsize=7)
ax[1].bar(range(n_pts), errors, color='tab:red')
ax[1].axhline(mean(errors), color='k', ls='--', lw=0.8,
              label='Mean error')
ax[1].set_xlabel('Precision point i')
ax[1].set_ylabel('Error $e_i$ (mm)')
ax[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

Desired trajectory (first 5 points):
  i        xd        yd
  0    150.00    100.00
  1    143.83    114.24
  2    134.20    124.85
  3    122.72    131.27
  4    110.91    133.58
r1=80.0, r2=40.0, r3=100.0, r4=90.0 mm
Coupler: xp=30.0, yp=10.0 mm; ground pivot: (60.0, 80.0) mm
Maximum loop-closure residual: 3.55e-14 mm
Mean trajectory error: 38.7818 mm
Maximum error:       53.0082 mm